In [1]:
!pip install -q datasets==3.6.0
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q transformers peft accelerate bitsandbytes
!pip install tree-sitter
!pip install tree-sitter-java
!pip install tree-sitter-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.4/635.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 11.3 MB/s eta 0:00:00


In [28]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, PeftModel
from google.colab import userdata
import datasets
from datasets import load_dataset, Dataset
import torch
import pandas as pd
import tree_sitter
import ast
import json
from tree_sitter import Language, Parser
import tree_sitter_java
import tree_sitter_python
import subprocess
import tempfile
import os

In [3]:
print(datasets.__version__)

3.6.0


In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

HF_TOKEN = userdata.get('HF_TOKEN')
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "bigcode/starcoder2-3b"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(model_name, token=HF_TOKEN, quantization_config=bnb_config).to(device)

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [5]:
!unzip -o /content/starcoder2-python-java-custom-lora.zip -d /content/starcoder2-python-java-custom-lora/

Archive:  /content/starcoder2-python-java-custom-lora.zip
  inflating: /content/starcoder2-python-java-custom-lora/adapter_config.json  
  inflating: /content/starcoder2-python-java-custom-lora/tokenizer.json  
  inflating: /content/starcoder2-python-java-custom-lora/adapter_model.safetensors  
  inflating: /content/starcoder2-python-java-custom-lora/training_args.bin  
  inflating: /content/starcoder2-python-java-custom-lora/tokenizer_config.json  
  inflating: /content/starcoder2-python-java-custom-lora/README.md  


In [6]:
# Load the LoRA adapters from the checkpoint
model_to_test = PeftModel.from_pretrained(model, "/content/starcoder2-python-java-custom-lora")

# Set the model to evaluation mode and move to device
model_to_test = model_to_test.eval().to(device)

print("Model loaded successfully for testing.")

Model loaded successfully for testing.


In [7]:
def generate_solution(prompt, num_solutions=1):
    full_prompt = f"""
             ### Instruction
             {prompt}
             ### Response
             """

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    # #Greedy decoding
    # outputs = model_to_test.generate(
    # **inputs,
    # max_new_tokens=300,
    # do_sample=False,
    #repetition_penalty=1.2,
    #num_return_sequences=num_solutions,
    # eos_token_id=tokenizer.eos_token_id,
    # pad_token_id=tokenizer.eos_token_id

    # )

    #Low temperature
    outputs = model_to_test.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        #repetition_penalty=1.2,
        num_return_sequences=num_solutions,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    # for pass@10, 20
    # outputs = model_to_test.generate(
    #     **inputs,
    #     max_new_tokens=300,
    #     do_sample=True, # Enable sampling for diverse solutions
    #     temperature=0.7, # Add temperature for diversity
    #     num_return_sequences=num_solutions, # Generate multiple solutions
    #     eos_token_id=tokenizer.eos_token_id
    # )

    # Decode all generated sequences
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    return generated_texts

In [8]:
def extract_code(text):
    if "### Response" in text:
        return text.split("### Response")[-1].strip()

    return text.strip()

def run_mbpp_tests(code, test_list):
    namespace = {}

    try:
        exec(code, namespace)

        for test in test_list:
            exec(test, namespace)

        return True

    except Exception:
        return False

In [9]:
JAVA_LANGUAGE = Language(
    tree_sitter_java.language()
)

parser = Parser()
parser.language = JAVA_LANGUAGE

In [10]:
def extract_java_ir(code):

    tree = parser.parse(
        bytes(code, "utf8")
    )

    root = tree.root_node

    ir = {
        "functions": 0,
        "for_loops": 0,
        "while_loops": 0,
        "ifs": 0,
        "returns": 0,
        "assignments": 0,
        "binary_ops": [],
        "comparisons": [],
        "calls": [],
        "data_structures": [],
        "recursion": False
    }

    method_names = set()

    def walk(node):

        if node.type == "method_declaration":

            ir["functions"] += 1

            for child in node.children:

                if child.type == "identifier":
                    method_names.add(
                        child.text.decode()
                    )

        elif node.type in [
            "for_statement",
            "enhanced_for_statement"
        ]:
            ir["for_loops"] += 1

        elif node.type == "while_statement":
            ir["while_loops"] += 1

        elif node.type == "if_statement":
            ir["ifs"] += 1

        elif node.type == "return_statement":
            ir["returns"] += 1

        elif node.type == "assignment_expression":
            ir["assignments"] += 1

        elif node.type == "method_invocation":

            for child in node.children:

                if child.type == "identifier":

                    name = child.text.decode()

                    ir["calls"].append(name)

                    if name in method_names:
                        ir["recursion"] = True

                    break

        elif node.type in [
            "+",
            "-",
            "*",
            "/",
            "%",
            "==",
            "!=",
            "<",
            ">",
            "<=",
            ">="
        ]:
            ir["binary_ops"].append(
                node.type
            )

        elif node.type == "array_creation_expression":
            ir["data_structures"].append(
                "array"
            )

        for child in node.children:
            walk(child)

    walk(root)

    for key in [
        "binary_ops",
        "comparisons",
        "calls",
        "data_structures"
    ]:
        ir[key] = sorted(
            list(set(ir[key]))
        )

    return ir

In [11]:
import ast

def extract_python_ir(code):

    tree = ast.parse(code)

    ir = {
        "functions": 0,
        "for_loops": 0,
        "while_loops": 0,
        "ifs": 0,
        "returns": 0,
        "assignments": 0,
        "binary_ops": [],
        "comparisons": [],
        "calls": [],
        "data_structures": [],
        "recursion": False
    }

    function_names = set()

    class Visitor(ast.NodeVisitor):

        def visit_FunctionDef(self, node):

            ir["functions"] += 1
            function_names.add(node.name)

            self.generic_visit(node)

        def visit_For(self, node):
            ir["for_loops"] += 1
            self.generic_visit(node)

        def visit_While(self, node):
            ir["while_loops"] += 1
            self.generic_visit(node)

        def visit_If(self, node):
            ir["ifs"] += 1
            self.generic_visit(node)

        def visit_Return(self, node):
            ir["returns"] += 1
            self.generic_visit(node)

        def visit_Assign(self, node):
            ir["assignments"] += 1
            self.generic_visit(node)

        def visit_Call(self, node):

            if isinstance(node.func, ast.Name):

                ir["calls"].append(node.func.id)

                if node.func.id in function_names:
                    ir["recursion"] = True

            self.generic_visit(node)

        def visit_BinOp(self, node):

            op = type(node.op).__name__

            ir["binary_ops"].append(op)

            self.generic_visit(node)

        def visit_Compare(self, node):

            for op in node.ops:
                ir["comparisons"].append(
                    type(op).__name__
                )

            self.generic_visit(node)

        def visit_List(self, node):
            ir["data_structures"].append("list")
            self.generic_visit(node)

        def visit_Dict(self, node):
            ir["data_structures"].append("dict")
            self.generic_visit(node)

        def visit_Set(self, node):
            ir["data_structures"].append("set")
            self.generic_visit(node)

        def visit_Tuple(self, node):
            ir["data_structures"].append("tuple")
            self.generic_visit(node)

    Visitor().visit(tree)

    for key in [
        "binary_ops",
        "comparisons",
        "calls",
        "data_structures"
    ]:
        ir[key] = sorted(list(set(ir[key])))

    return ir

In [12]:
def similarity(ir1, ir2):

    score = 0
    total = 0

    numeric_fields = [
        "functions",
        "for_loops",
        "while_loops",
        "ifs",
        "returns",
        "assignments"
    ]

    for field in numeric_fields:

        total += 1

        if ir1[field] == ir2[field]:
            score += 1

    list_fields = [
        "binary_ops",
        "comparisons",
        "calls",
        "data_structures"
    ]

    for field in list_fields:

        total += 1

        s1 = set(ir1[field])
        s2 = set(ir2[field])

        union = s1.union(s2)

        if len(union) == 0:
            score += 1
        else:
            score += (
                len(s1.intersection(s2))
                / len(union)
            )

    total += 1

    if ir1["recursion"] == ir2["recursion"]:
        score += 1

    return round(score / total, 4)

In [13]:
def compare_java_python(java_problem, python_code):
    java_ir = extract_java_ir(java_problem)
    python_ir = extract_python_ir(python_code)

    score = similarity(
        java_ir,
        python_ir
    )
    print(f"Similarity: {score}")
    return score

In [14]:
translation_pairs_df = pd.read_csv('/content/java_python_translation_pairs_corrected.csv')
display(translation_pairs_df.head())

,qid,text,java_code,python_code
0,602,Find the first repeated character in a given s...,import java.io.*;\nimport java.lang.*;\nimport...,"def first_repeated_char(str1):\n for index,c ..."
1,603,get a lucid number smaller than or equal to n.,import java.io.*;\nimport java.lang.*;\nimport...,def get_ludic(n):\n\tludics = []\n\tfor i in r...
2,604,reverse words in a given string.,import java.io.*;\nimport java.lang.*;\nimport...,def reverse_words(s):\n return ' '.join...
3,605,check if the given integer is a prime number.,import java.io.*;\nimport java.lang.*;\nimport...,def prime_num(num):\n if num >=1:\n for i i...
4,606,convert degrees to radians.,import java.io.*;\nimport java.lang.*;\nimport...,import math\ndef radian_degree(degree):\n radi...


In [ ]:
results_data = []

for index, row in translation_pairs_df.iterrows():
    java_problem_text = f"""
    convert the following java code to python
    {row['java_code']}
    """
    java_problem = row['java_code']
    python_solution_original = row['python_code'] # Keep original Python solution

    # Generate Python solution
    generated_solutions = generate_solution(java_problem_text, num_solutions=1)
    generated_python_code = extract_code(generated_solutions[0])

    # Calculate similarity score
    try:
        score = compare_java_python(java_problem, generated_python_code)
    except Exception as e:
        score = 0.0 # Assign a default score if an error occurs during comparison
        print(f"Error calculating similarity for qid {row['qid']}: {e}")

    # Collect all data for the new DataFrame
    results_data.append({
        'qid': row['qid'],
        'text': row['text'], # Include original 'text' column if desired
        'java_code': java_problem,
        'python_code_original': python_solution_original,
        'generated_python_code': generated_python_code,
        'ast_similarity_score': score
    })

results_df = pd.DataFrame(results_data)

print("Processing complete. Displaying the new DataFrame with results:")
display(results_df.head())

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Similarity: 0.6364
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.7273
Similarity: 0.7273
Similarity: 0.8182
Similarity: 0.6364
Similarity: 0.4545
Similarity: 0.6364
Similarity: 0.7273
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.4545
Similarity: 0.6364
Similarity: 0.4545
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.3636
Similarity: 0.8182
Similarity: 0.8182
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.8182
Similarity: 0.3636
Similarity: 0.8182
Similarity: 0.7273
Similarity: 0.5455
Similarity: 0.8182
Similarity: 0.6364
Similarity: 0.8182
Similarity: 0.8182
Similarity: 0.6364
Similarity: 0.4545
Similarity: 0.5455
Similarity: 0.5455
Similarity: 0.5455
Similarity: 0.8182
Similarity: 0.5455
Similarity: 0.4545
Similarity: 0.8182
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.4545
Similarity: 0.8182
Similarity: 0.7273
Similarity: 0.5455
Similarity: 0.7273
Similarity: 0.4545
Similarity: 

<unknown>:3: SyntaxWarning: invalid escape sequence '\.'


Similarity: 0.5455
Similarity: 0.5455
Similarity: 0.7273
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.4545
Similarity: 0.6364
Similarity: 0.8182
Similarity: 0.7273
Similarity: 0.4545
Similarity: 0.8312
Similarity: 0.4545
Similarity: 0.7273
Similarity: 0.6364
Similarity: 0.5455
Similarity: 0.4545
Similarity: 0.4545
Similarity: 0.4545
Similarity: 0.5455
Similarity: 0.8182
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.4545
Similarity: 0.4545
Similarity: 0.6364
Similarity: 0.7273
Similarity: 0.4545
Similarity: 0.7273
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.8182
Similarity: 0.5455
Similarity: 0.8182
Similarity: 0.5455
Similarity: 0.4545
Similarity: 0.6364
Similarity: 0.6667
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.7273
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.8182
Similarity: 0.3636
Similarity: 0.6364
Similarity: 0.5455
Similarity: 0.5455
Similarity: 0.4773
Similarity: 

,qid,text,java_code,python_code_original,generated_python_code,ast_similarity_score
0,602,Find the first repeated character in a given s...,import java.io.*;\nimport java.lang.*;\nimport...,"def first_repeated_char(str1):\n for index,c ...",def first_repeated_char(str1):\n for i in ran...,0.6364
1,603,get a lucid number smaller than or equal to n.,import java.io.*;\nimport java.lang.*;\nimport...,def get_ludic(n):\n\tludics = []\n\tfor i in r...,"def get_ludic(n): \n ludics = [1, 2, 3, 5, ...",0.5455
2,604,reverse words in a given string.,import java.io.*;\nimport java.lang.*;\nimport...,def reverse_words(s):\n return ' '.join...,def reverse_words(s):\n return ' '.join(rev...,0.6364
3,605,check if the given integer is a prime number.,import java.io.*;\nimport java.lang.*;\nimport...,def prime_num(num):\n if num >=1:\n for i i...,def prime_num(num):\n if num == 1:\n ...,0.7273
4,606,convert degrees to radians.,import java.io.*;\nimport java.lang.*;\nimport...,import math\ndef radian_degree(degree):\n radi...,def radian_degree(degree):\n radian = (degree...,0.7273


### Calculating CodeBLEU, CodeBERT, and ROUGE scores

In [15]:
!pip install -q evaluate sentence-transformers nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


In [16]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=616a3144429248ea714a255bb0e73ec921e412975f1cd1021397a76e29460bfa
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [17]:
import evaluate
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk

# Download necessary NLTK data for BLEU (if not already present)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Load BLEU and ROUGE metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# Load a SentenceTransformer model for CodeBERT-like embeddings
# Using a general-purpose model; for true CodeBERT, a code-specific model would be better if available via SentenceTransformers
# For better performance on code, consider models like 'sentence-transformers/all-distilroberta-v1' or a fine-tuned code model.
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def calculate_code_metrics(reference_code, generated_code):
    # BLEU score
    # The 'evaluate' library expects lists of strings for references and predictions
    # For a single pair, we wrap them in lists
    bleu_score = bleu.compute(predictions=[generated_code], references=[[reference_code]])

    # ROUGE score
    rouge_score = rouge.compute(predictions=[generated_code], references=[reference_code])

    # CodeBERT-like similarity using sentence embeddings
    # Generate embeddings for both original and generated code
    embeddings_ref = embedding_model.encode(reference_code, convert_to_tensor=True)
    embeddings_gen = embedding_model.encode(generated_code, convert_to_tensor=True)

    # Calculate cosine similarity
    # Reshape for sklearn's cosine_similarity if they are 1D tensors/arrays
    if embeddings_ref.dim() == 1:
        embeddings_ref = embeddings_ref.unsqueeze(0)
    if embeddings_gen.dim() == 1:
        embeddings_gen = embeddings_gen.unsqueeze(0)

    codebert_similarity = cosine_similarity(embeddings_ref.cpu(), embeddings_gen.cpu())[0][0]

    return {
        'bleu': bleu_score['bleu'],
        'rouge1': rouge_score['rouge1'],
        'rouge2': rouge_score['rouge2'],
        'rougeL': rouge_score['rougeL'],
        'codebert_similarity': float(codebert_similarity)
    }

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Initialize new columns
results_df['bleu_score'] = 0.0
results_df['rouge1_score'] = 0.0
results_df['rouge2_score'] = 0.0
results_df['rougeL_score'] = 0.0
results_df['codebert_similarity_score'] = 0.0

print("Calculating additional metrics for results_df...")

# Iterate and calculate scores, updating the DataFrame
for idx, row in results_df.iterrows():
    ref_code = row['python_code_original']
    gen_code = row['generated_python_code']

    metrics = calculate_code_metrics(ref_code, gen_code)

    results_df.at[idx, 'bleu_score'] = metrics['bleu']
    results_df.at[idx, 'rouge1_score'] = metrics['rouge1']
    results_df.at[idx, 'rouge2_score'] = metrics['rouge2']
    results_df.at[idx, 'rougeL_score'] = metrics['rougeL']
    results_df.at[idx, 'codebert_similarity_score'] = metrics['codebert_similarity']

print("Metrics calculation complete. Displaying updated DataFrame head:")
display(results_df.head())

NameError: name 'results_df' is not defined

### Calculating Composite Translation Score

In [ ]:
results_df["translation_score"] = (
    0.35 * results_df["codebert_similarity_score"] +
    0.25 * results_df["ast_similarity_score"] +
    0.15 * results_df["rougeL_score"] +
    0.10 * results_df["rouge1_score"] +
    0.05 * results_df["rouge2_score"] +
    0.10 * results_df["bleu_score"]
)

print("Composite translation score calculated with new weights. Displaying updated DataFrame head:")
display(results_df.head())

Composite translation score calculated with new weights. Displaying updated DataFrame head:


,qid,text,java_code,python_code_original,generated_python_code,similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score,translation_score
0,602,Find the first repeated character in a given s...,import java.io.*;\nimport java.lang.*;\nimport...,"def first_repeated_char(str1):\n for index,c ...",def first_repeated_char(str1):\n for i in r...,0.7273,0.491468,0.681818,0.428571,0.681818,0.932166,0.749113
1,603,get a lucid number smaller than or equal to n.,import java.io.*;\nimport java.lang.*;\nimport...,def get_ludic(n):\n\tludics = []\n\tfor i in r...,"def get_ludic(n): \n ludics = [1, 2, 3, 5, ...",0.4545,0.247624,0.382022,0.252874,0.359551,0.850332,0.540782
2,604,reverse words in a given string.,import java.io.*;\nimport java.lang.*;\nimport...,def reverse_words(s):\n return ' '.join...,def reverse_words(s):\n return ' '.join(rev...,0.6364,1.000000,1.000000,1.000000,1.000000,1.000000,0.909100
3,605,check if the given integer is a prime number.,import java.io.*;\nimport java.lang.*;\nimport...,def prime_num(num):\n if num >=1:\n for i i...,def prime_num(num):\n if num == 1:\n retur...,0.7273,0.366605,0.705882,0.448980,0.509804,0.960581,0.724197
4,606,convert degrees to radians.,import java.io.*;\nimport java.lang.*;\nimport...,import math\ndef radian_degree(degree):\n radi...,def radian_degree(degree):\n radian = degree ...,0.7273,0.909156,0.916667,0.909091,0.916667,0.964735,0.885019


### Mean Scores

In [ ]:
mean_translation_score = results_df['translation_score'].mean()
mean_codebert_similarity = results_df['codebert_similarity_score'].mean()
mean_ast_similarity = results_df['similarity_score'].mean()
mean_rougeL = results_df['rougeL_score'].mean()
mean_rouge1 = results_df['rouge1_score'].mean()
mean_rouge2 = results_df['rouge2_score'].mean()
mean_bleu = results_df['bleu_score'].mean()

print(f"Mean Translation Score: {mean_translation_score:.4f}")
print(f"Mean CodeBERT Similarity Score: {mean_codebert_similarity:.4f}")
print(f"Mean AST Similarity Score: {mean_ast_similarity:.4f}")
print(f"Mean ROUGE-L Score: {mean_rougeL:.4f}")
print(f"Mean ROUGE-1 Score: {mean_rouge1:.4f}")
print(f"Mean ROUGE-2 Score: {mean_rouge2:.4f}")
print(f"Mean BLEU Score: {mean_bleu:.4f}")

Mean Translation Score: 0.6824
Mean CodeBERT Similarity Score: 0.9180
Mean AST Similarity Score: 0.6000
Mean ROUGE-L Score: 0.5794
Mean ROUGE-1 Score: 0.6262
Mean ROUGE-2 Score: 0.4454
Mean BLEU Score: 0.3929


### Python to Java Translation and Metric Calculation

First, let's define a function to generate Java code from a Python problem description using our fine-tuned model.

In [38]:
def generate_java_solution(prompt, num_solutions=1):
    full_prompt = f"""
             ### Instruction
             Convert the following Python code to Java:
             {prompt}
             ### Response
             """

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    outputs = model_to_test.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        num_return_sequences=num_solutions,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

    return generated_texts

Next, we'll need a function to compare two Java codes using their Abstract Syntax Trees (ASTs).

In [37]:
def compare_java_java(java_code1, java_code2):
    java_ir1 = extract_java_ir(java_code1)
    java_ir2 = extract_java_ir(java_code2)

    score = similarity(
        java_ir1,
        java_ir2
    )
    # print(f"Similarity: {score}")
    return score

In [36]:
def check_java_compilation(java_code):
    # Create a temporary directory
    with tempfile.TemporaryDirectory() as tmpdir:
        # Write the Java code to a .java file
        file_path = os.path.join(tmpdir, "Solution.java")
        with open(file_path, "w") as f:
            f.write(java_code)

        # Try to compile the Java file
        try:
            # Use subprocess to run javac
            compile_process = subprocess.run(
                ["javac", file_path],
                capture_output=True,
                text=True,
                check=False # Do not raise an exception for non-zero exit codes
            )
            # If javac returns 0, compilation was successful
            if compile_process.returncode == 0:
                return True
            else:
                # print(f"Compilation error: {compile_process.stderr}")
                return False
        except FileNotFoundError:
            print("Error: javac command not found. Make sure Java Development Kit (JDK) is installed and in your PATH.")
            return False
        except Exception as e:
            print(f"An unexpected error occurred during compilation check: {e}")
            return False

Now, let's run the evaluation loop for Python to Java translation, calculate all specified metrics, and store them in a new DataFrame.

In [39]:
java_translation_results_data = []

for index, row in translation_pairs_df.iterrows():
    python_problem = row['python_code']
    java_solution_original = row['java_code'] # Reference Java solution

    # Generate Java solution
    generated_solutions = generate_java_solution(python_problem, num_solutions=1)
    generated_java_code = extract_code(generated_solutions[0])

    # Check Java compilation rate
    is_compilable = check_java_compilation(generated_java_code)

    # Calculate AST similarity score between generated Java and original Java
    try:
        ast_score = compare_java_java(java_solution_original, generated_java_code)
    except Exception as e:
        ast_score = 0.0 # Assign a default score if an error occurs during comparison
        print(f"Error calculating AST similarity for qid {row['qid']}: {e}")

    # Calculate other code metrics (BLEU, ROUGE, CodeBERT-like) between generated Java and original Java
    try:
        code_metrics = calculate_code_metrics(java_solution_original, generated_java_code)
    except Exception as e:
        code_metrics = {
            'bleu': 0.0,
            'rouge1': 0.0,
            'rouge2': 0.0,
            'rougeL': 0.0,
            'codebert_similarity': 0.0
        }
        print(f"Error calculating other metrics for qid {row['qid']}: {e}")


    # Collect all data for the new DataFrame
    java_translation_results_data.append({
        'qid': row['qid'],
        'text': row['text'],
        'python_code_original': python_problem,
        'java_code_original': java_solution_original,
        'generated_java_code': generated_java_code,
        'is_compilable': is_compilable,
        'ast_similarity_score': ast_score,
        'bleu_score': code_metrics['bleu'],
        'rouge1_score': code_metrics['rouge1'],
        'rouge2_score': code_metrics['rouge2'],
        'rougeL_score': code_metrics['rougeL'],
        'codebert_similarity_score': code_metrics['codebert_similarity']
    })

java_translation_results_df = pd.DataFrame(java_translation_results_data)

print("Processing complete. Displaying the new DataFrame with Python-to-Java translation results:")
display(java_translation_results_df.head())

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

Processing complete. Displaying the new DataFrame with Python-to-Java translation results:


,qid,text,python_code_original,java_code_original,generated_java_code,is_compilable,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score
0,602,Find the first repeated character in a given s...,"def first_repeated_char(str1):\n for index,c ...",import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.7273,0.530798,0.675862,0.475524,0.620690,0.879773
1,603,get a lucid number smaller than or equal to n.,def get_ludic(n):\n\tludics = []\n\tfor i in r...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,False,0.3864,0.326592,0.457711,0.371859,0.447761,0.851839
2,604,reverse words in a given string.,def reverse_words(s):\n return ' '.join...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.9394,0.803332,0.781955,0.656489,0.781955,0.923949
3,605,check if the given integer is a prime number.,def prime_num(num):\n if num >=1:\n for i i...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.4935,0.657239,0.743802,0.504202,0.628099,0.927762
4,606,convert degrees to radians.,import math\ndef radian_degree(degree):\n radi...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.8636,0.705349,0.774194,0.593407,0.731183,0.917448


### Composite Translation Score for Python to Java

In [40]:
java_translation_results_df["translation_score"] = (
    0.35 * java_translation_results_df["codebert_similarity_score"] +
    0.25 * java_translation_results_df["ast_similarity_score"] +
    0.15 * java_translation_results_df["rougeL_score"] +
    0.10 * java_translation_results_df["rouge1_score"] +
    0.05 * java_translation_results_df["rouge2_score"] +
    0.10 * java_translation_results_df["bleu_score"]
)

print("Composite translation score calculated for Python to Java. Displaying updated DataFrame head:")
display(java_translation_results_df.head(10))

Composite translation score calculated for Python to Java. Displaying updated DataFrame head:


,qid,text,python_code_original,java_code_original,generated_java_code,is_compilable,ast_similarity_score,bleu_score,rouge1_score,rouge2_score,rougeL_score,codebert_similarity_score,translation_score
0,602,Find the first repeated character in a given s...,"def first_repeated_char(str1):\n for index,c ...",import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.7273,0.530798,0.675862,0.475524,0.620690,0.879773,0.727291
1,603,get a lucid number smaller than or equal to n.,def get_ludic(n):\n\tludics = []\n\tfor i in r...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,False,0.3864,0.326592,0.457711,0.371859,0.447761,0.851839,0.558931
2,604,reverse words in a given string.,def reverse_words(s):\n return ' '.join...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.9394,0.803332,0.781955,0.656489,0.781955,0.923949,0.866879
3,605,check if the given integer is a prime number.,def prime_num(num):\n if num >=1:\n for i i...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.4935,0.657239,0.743802,0.504202,0.628099,0.927762,0.707621
4,606,convert degrees to radians.,import math\ndef radian_degree(degree):\n radi...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.8636,0.705349,0.774194,0.593407,0.731183,0.917448,0.824309
5,608,Find nth bell number.,def bell_Number(n): \n bell = [[0 for i in ...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.8182,0.640497,0.730435,0.530973,0.660870,0.953817,0.801158
6,609,Find minimum possible value for the given peri...,"def floor_Min(A,B,N):\n x = max(B - 1,N)\n ...",import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.6545,0.303880,0.552147,0.310559,0.478528,0.740523,0.595718
7,610,Remove the k'th element from a given list.,"def remove_kth_element(list1, L):\n return ...",import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,False,0.4935,0.373467,0.611321,0.441065,0.505660,0.936592,0.647563
8,611,find the maximum of nth column from the given...,"def max_of_nth(test_list, N):\n res = max([su...",import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.9091,0.547001,0.763819,0.467005,0.673367,0.883258,0.791853
9,614,find the cumulative sum of all the values tha...,def cummulative_sum(test_list):\n res = sum(m...,import java.io.*;\nimport java.lang.*;\nimport...,import java.io.*;\nimport java.lang.*;\nimport...,True,0.9091,0.524334,0.628272,0.433862,0.586387,0.929318,0.777448


### Mean Scores for Python to Java Translation

In [41]:
mean_translation_score_java = java_translation_results_df['translation_score'].mean()
mean_codebert_similarity_java = java_translation_results_df['codebert_similarity_score'].mean()
mean_ast_similarity_java = java_translation_results_df['ast_similarity_score'].mean()
mean_rougeL_java = java_translation_results_df['rougeL_score'].mean()
mean_rouge1_java = java_translation_results_df['rouge1_score'].mean()
mean_rouge2_java = java_translation_results_df['rouge2_score'].mean()
mean_bleu_java = java_translation_results_df['bleu_score'].mean()

print(f"Mean Python to Java Translation Score: {mean_translation_score_java:.4f}")
print(f"Mean Python to Java CodeBERT Similarity Score: {mean_codebert_similarity_java:.4f}")
print(f"Mean Python to Java AST Similarity Score: {mean_ast_similarity_java:.4f}")
print(f"Mean Python to Java ROUGE-L Score: {mean_rougeL_java:.4f}")
print(f"Mean Python to Java ROUGE-1 Score: {mean_rouge1_java:.4f}")
print(f"Mean Python to Java ROUGE-2 Score: {mean_rouge2_java:.4f}")
print(f"Mean Python to Java BLEU Score: {mean_bleu_java:.4f}")

Mean Python to Java Translation Score: 0.7122
Mean Python to Java CodeBERT Similarity Score: 0.8797
Mean Python to Java AST Similarity Score: 0.7011
Mean Python to Java ROUGE-L Score: 0.5929
Mean Python to Java ROUGE-1 Score: 0.6503
Mean Python to Java ROUGE-2 Score: 0.4695
Mean Python to Java BLEU Score: 0.5161


In [42]:
compilation_rate = (java_translation_results_df['is_compilable'].sum() / len(java_translation_results_df)) * 100
print(f"Java Compilation Rate: {compilation_rate:.2f}%")

Java Compilation Rate: 70.48%


In [44]:
java_translation_results_df.to_csv('python_to_java_translation_results.csv', index=False)
print("DataFrame saved to 'python_to_java_translation_results.csv'")

DataFrame saved to 'python_to_java_translation_results.csv'
